# Hybrid depth: artistic layers **and** continuous depth

<a href="https://colab.research.google.com/github/saranunt-0/layer-mapping/blob/main/notebooks/03_hybrid_depth_poc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Illustrator's Depth gives clean, artistically meaningful **layers**, but each layer is flat: a road or a plain running
to the horizon becomes one constant value. A true depth model (Depth Anything V2) gives **smooth** depth, but its layer
edges are soft and sometimes in the wrong order on paintings. We want both:

* inside a layer, depth varies smoothly (the road recedes)
* between layers, depth jumps, and the jump agrees with the paint order

| Part | What | Licence |
|---|---|---|
| 1 | Run Illustrator's Depth + Depth Anything, then **combine them with classical methods** (multiply, blend, stacked, ordered offsets, Poisson, guided filter, masked re-prediction) | Uses Adobe's model: **non-commercial research only** |
| 2 | **One model** (Depth Anything V2 Small, two output channels) trained to produce hybrid depth directly | Fully open: Apache-2.0 backbone + synthetic data |

Part 2 is trained on **synthetic scenes with exact ground truth**, never on Illustrator's Depth outputs. Training on
those outputs would make your model a derivative of a non-commercial model (see `docs/PLAN.md` §3).

Runtime: GPU (T4 is enough), about 60-75 min. Set `USE_ILLUSTRATORS_DEPTH = False` to skip Adobe's model entirely.

## 0. Setup

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/saranunt-0/layer-mapping.git"
BRANCH = "main"
REPO_DIR = "/content/layer-mapping"

if not os.path.exists(REPO_DIR):
    url = REPO_URL
    try:
        # Only needed while the repo is private: add a GITHUB_TOKEN secret (key icon, left sidebar).
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
sys.path.insert(0, REPO_DIR)

import torch, transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.45"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers"], check=True)
    raise SystemExit("transformers was upgraded: Runtime > Restart session, then run all cells again.")
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - Runtime > Change runtime type > GPU")
print("CPU cores:", os.cpu_count())

In [ ]:
import json, time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from layermap import data, layers, metrics, synth
from layermap import model as lm
from layermap import train as lt


def load_rgb(path, max_side=1024):
    rgb = cv2.imread(str(path), cv2.IMREAD_COLOR)[..., ::-1]
    s = max_side / max(rgb.shape[:2])
    if s < 1:
        rgb = cv2.resize(rgb, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    return np.ascontiguousarray(rgb)


def show_row(images, titles, size=4.0):
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size))
    for ax, im, t in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(im)
        ax.set_title(t, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def mean_metrics(rows):
    return {k: round(float(np.nanmean([r[k] for r in rows])), 4) for k in rows[0]}


def prior_metrics(ds):
    """Score of the "lower in the frame = closer" rule on an eval dataset (per target if several)."""
    names = getattr(ds, "targets", ("layer",))
    rows = []
    for i in range(len(ds)):
        t = ds[i][1]
        t = t[None] if t.dim() == 2 else t
        row = {}
        for c, n in enumerate(names):
            pre = "" if len(names) == 1 else f"{n}/"
            row.update({pre + k: v for k, v in metrics.all_metrics(metrics.vertical_prior(t[c].shape), t[c]).items()})
        rows.append(row)
    return mean_metrics(rows)


import torch
from IPython.display import Image as IPImage, display

from layermap import fusion, hybrid_eval, viz
from layermap.metrics import hybrid_metrics


def summarize(rows):
    """Mean of each metric per method over a list of {method: metrics} dicts, as a DataFrame."""
    import pandas as pd

    methods = list(dict.fromkeys(k for r in rows for k in r))
    table = {m: {q: np.nanmean([r[m][q] for r in rows if m in r]) for q in next(r[m] for r in rows if m in r)}
             for m in methods}
    return pd.DataFrame(table).T.round(3)


def plot_profiles(maps, labels, title=""):
    col, prof = hybrid_eval.depth_profile(maps, labels)
    plt.figure(figsize=(7, 3))
    for k, v in prof.items():
        plt.plot(v, label=k, lw=1.5)
    plt.xlabel(f"image row (top -> bottom), column {col}"); plt.ylabel("normalised depth (1 = near)")
    plt.title(title, fontsize=9); plt.legend(fontsize=7); plt.show()

In [ ]:
USE_ILLUSTRATORS_DEPTH = True  # False: Part 1 uses a k-means stand-in layer map instead
N_ID_LAYERS = 8                # discrete layers taken from Illustrator's Depth
ID_BLUR = 1.5                  # its README suggests blurring textured inputs (canvas, paper)
MARGIN = 0.02                  # minimum depth jump between touching layers (0 = pure physical depth)
INFER_SIZE = 518

In [ ]:
# Illustrator's Depth (Adobe Research License: NON-COMMERCIAL RESEARCH ONLY).
# Cloned and downloaded at runtime; nothing from it is stored in this repository.
ID_DIR = "/content/illustrators_depth"
ID_CKPT = f"{ID_DIR}/checkpoints/mmsvg_model/checkpoints/id_model.ckpt"
id_model = None
if USE_ILLUSTRATORS_DEPTH:
    try:
        if not os.path.exists(ID_DIR):
            subprocess.run(["git", "clone", "-q", "--recursive", "https://github.com/nissmar/illustrators_depth",
                            ID_DIR], check=True)
        # Depth Pro pins numpy<2, so it is put on sys.path instead of being pip-installed.
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightning==2.5.3", "svgpathtools",
                        "pillow_heif", "timm", "gdown"], check=True)
        if not os.path.exists(ID_CKPT):
            os.makedirs(f"{ID_DIR}/checkpoints", exist_ok=True)
            subprocess.run(["gdown", "-q", "12BRCnHIQutSThfJhhHn_sYOBxFeq-NB3", "-O",
                            f"{ID_DIR}/checkpoints/mmsvg_model.zip"], check=True)
            subprocess.run(["unzip", "-q", "-o", f"{ID_DIR}/checkpoints/mmsvg_model.zip", "-d",
                            f"{ID_DIR}/checkpoints"], check=True)
        for p in (ID_DIR, f"{ID_DIR}/ml-depth-pro/src"):
            if p not in sys.path:
                sys.path.insert(0, p)
        # The Lightning checkpoint stores its config as pickled Python objects; this trusts that one file.
        os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
        from src.model.illustrators_depth_model import load_illustrators_depth_model
        id_model, _ = load_illustrators_depth_model(ID_CKPT, device="cuda" if torch.cuda.is_available() else "cpu")
        print("Illustrator's Depth loaded")
    except Exception as e:
        print("Illustrator's Depth unavailable, falling back to a stand-in layer map:", repr(e))


def id_predict(rgb):
    """Raw Illustrator's Depth at the image resolution (larger = painted later = in front)."""
    from PIL import Image
    with torch.no_grad():
        _, pred = id_model.infer_single_image(Image.fromarray(rgb), preserve_size=False, gaussian_sigma=ID_BLUR)
    return cv2.resize(np.asarray(pred, np.float32), (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_LINEAR)


def layer_map(rgb, depth):
    """Discrete layers from Illustrator's Depth, or (stand-in) from k-means on Depth Anything."""
    if id_model is not None:
        raw = id_predict(rgb)
        return raw, layers.depth_to_layers(raw, N_ID_LAYERS)
    return None, layers.depth_to_layers(depth, N_ID_LAYERS, rgb=rgb)

In [ ]:
base = lm.load_model(lm.DEFAULT_BACKBONE)  # Depth Anything V2 Small, the 'true depth' model

## 1. Paintings (public domain, CC0 images)

In [ ]:
import requests

PAINT_DIR = Path("/content/paintings")
PAINT_DIR.mkdir(exist_ok=True)
# Roads and open plains are the cases this notebook is about.
QUERIES = ["Caillebotte Paris Street Rainy Day", "Pissarro road", "Sisley road", "Monet wheat field",
           "Monet poplars", "Van Gogh landscape"]
PER_QUERY = 2
HEADERS = {"AIC-User-Agent": "layer-mapping PoC (non-commercial research)"}


def fetch_aic(query, n):
    """Public-domain paintings from the Art Institute of Chicago open-access API (images are CC0)."""
    r = requests.get("https://api.artic.edu/api/v1/artworks/search", headers=HEADERS, timeout=30, params={
        "q": query, "limit": 25, "query[term][is_public_domain]": "true",
        "fields": "id,title,artist_title,image_id,is_public_domain,artwork_type_title"})
    r.raise_for_status()
    got = []
    for a in r.json().get("data", []):
        if not (a.get("is_public_domain") and a.get("image_id")):
            continue
        if (a.get("artwork_type_title") or "Painting") != "Painting":
            continue
        out = PAINT_DIR / f"aic_{a['id']}.jpg"
        if not out.exists():
            img = requests.get(f"https://www.artic.edu/iiif/2/{a['image_id']}/full/843,/0/default.jpg",
                               headers=HEADERS, timeout=60)
            img.raise_for_status()
            out.write_bytes(img.content)
        got.append(out)
        if len(got) == n:
            break
    return got


for q in QUERIES:
    try:
        print(q, "->", [p.name for p in fetch_aic(q, PER_QUERY)])
    except Exception as e:
        print(q, "-> failed:", e)

In [ ]:
# Optional: add your own paintings. Set UPLOAD = True and pick files when the button appears.
UPLOAD = False
if UPLOAD:
    from google.colab import files
    for name, content in files.upload().items():
        (PAINT_DIR / name).write_bytes(content)

paintings = sorted(p for p in PAINT_DIR.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp"))
print(len(paintings), "paintings")
if not paintings:
    print("No paintings: the museum download failed and nothing was uploaded. Set UPLOAD = True and re-run.")

## 2. Part 1: classical fusion on real paintings

`layermap.fusion` takes a depth map `d` (larger = closer) and a layer map `L` (larger = in front):

| Method | Formula | Keeps road gradient | Guarantees layer order |
|---|---|---|---|
| multiply | `n(L) · n(d)` | yes, scaled per layer | no |
| blend | `a·n(L) + (1-a)·n(d)` | yes, compressed | no |
| stacked | `L + 0.8 · (d rescaled inside each layer)` | yes | **globally** (every layer in front of every earlier one) |
| ordered | `d + b[L]`, smallest offsets `b` that put each layer `MARGIN` in front of the layers it touches | **exactly** | where layers touch |
| poisson | least squares: gradients of `d` inside layers, ordered jumps across them | approximately | where layers touch |
| guided | guided filter of `d` with `L` as guide | smoothed | no (only sharpens edges) |
| masked+ordered | re-predict `d` for each layer with the layers in front removed, then `ordered` | yes | where layers touch |

`stacked` matches Illustrator's semantics (a tree painted over a meadow is in front of the *whole* meadow).
`ordered` keeps physical depth wherever it does not contradict the paint order, so the near part of the meadow stays
in front of a mid-distance tree. That is usually what a 3D/parallax effect needs.

In [ ]:
SHOW = ["multiply", "stacked", "ordered", "poisson"]
MASKED = True  # one extra depth prediction per layer
part1 = {}
for p in paintings:
    rgb = load_rgb(p, max_side=768)
    d = lm.predict(base, rgb, size=INFER_SIZE)
    id_raw, L = layer_map(rgb, d)
    fused = hybrid_eval.run_fusions(d, L, ordered={"margin": MARGIN}, poisson={"margin": MARGIN})
    if MASKED:
        dm = fusion.masked_depth(hybrid_eval.predictor(base, INFER_SIZE), rgb, L, base=d)
        fused["masked+ordered"] = fusion.fuse_ordered(dm, L, MARGIN)
    part1[p] = {"rgb": rgb, "depth": d, "id_raw": id_raw, "labels": L, **fused}
    first = [rgb, layers.colorize_depth(d), layers.colorize(L)]
    show_row(first + [layers.colorize_depth(fused[m]) for m in SHOW],
             [p.name, "Depth Anything", "layers (ID)" if id_model else "layers (stand-in)"] + SHOW, size=3)
    profile = {"Depth Anything": d, **({"Illustrator's Depth": id_raw} if id_raw is not None else {}),
               **{m: fused[m] for m in ("multiply", "stacked", "ordered")}}
    plot_profiles(profile, L, p.name)

**How to read the profiles:** one image column, top to bottom. A good hybrid rises *smoothly* along a road or field
(like Depth Anything) and *jumps* where a new layer starts (like Illustrator's Depth). Flat segments mean lost
continuity; a dip at a boundary means a layer is out of order.

## 3. Part 1, measured: synthetic scenes with exact ground truth

Paintings have no ground truth, so the numbers come from `layermap.synth` scenes. These are painterly
landscapes with roads and plains, and the generator knows both the paint-order layers and the true depth. The layer map
fed to the fusers is: the exact layers (**oracle**, an upper bound), the exact layers with wobbly boundaries and merged
layers (**noisy**, a realistic imperfect predictor), and, if loaded, **Illustrator's Depth** itself.

| Metric | Meaning |
|---|---|
| `acc_edge` | pixel pairs across layer boundaries ordered like the paint order (artistic layering) |
| `rho_within` | rank correlation with true depth *inside* layers (smooth road / plain) |
| `acc_hard` | pairs where the upper pixel is in front, vs the hybrid target |
| `ssi_mae` | distance to the hybrid target (`fusion.hybrid_target`) after scale/shift alignment |

In [ ]:
BENCH = [synth.render(s, synth.SynthConfig(size=518), return_depth=True) for s in range(70_000, 70_030)]
FUSE = ["multiply", "blend", "stacked", "ordered", "poisson", "guided"]
kw = {"ordered": {"margin": MARGIN}, "poisson": {"margin": MARGIN, "max_side": 384}}
bench_rows = []
for i, (rgb, lab, dep) in enumerate(BENCH):
    T = fusion.hybrid_target(dep, lab, MARGIN)
    d = lm.predict(base, rgb, size=INFER_SIZE, flip_tta=False)
    row = {"Depth Anything only": hybrid_metrics(d, T, dep, lab)}
    sources = {"oracle": lab, "noisy": hybrid_eval.degrade_labels(lab, seed=i)}
    if id_model is not None:
        raw = id_predict(rgb)
        row["Illustrator's Depth only"] = hybrid_metrics(raw, T, dep, lab)
        sources["ID"] = layers.depth_to_layers(raw, N_ID_LAYERS)
    for src, L in sources.items():
        for m, out in hybrid_eval.run_fusions(d, L, FUSE, **kw).items():
            row[f"{m} | {src}"] = hybrid_metrics(out, T, dep, lab)
    bench_rows.append(row)
    print(f"\r{i + 1}/{len(BENCH)}", end="")
part1_table = summarize(bench_rows)
part1_table

## 4. Part 2: one model that does both

The same Depth Anything V2 Small, with its last 1×1 convolution widened to **two output channels**:

* channel 0, `hybrid`: trained on `fusion.hybrid_target(true_depth, layers)`, i.e. continuous depth with ordered jumps
* channel 1, `layer`: trained on the paint-order layer index (gives discrete layers for editing, like Illustrator's Depth)

The losses are the same scale/shift-invariant L1 + gradient matching + pairwise ranking terms. Gradient matching is
what teaches "smooth inside, sharp at edges". Everything is trained on synthetic data with exact labels, so
there is no teacher model and no licence inheritance. About 8 min of data generation, then about 20 min of training on a T4.

In [ ]:
HDATA = Path("/content/data_hybrid")
cfg_h = synth.SynthConfig(size=518)
data.generate_dataset(HDATA / "train", 2000, 0, cfg_h, with_depth=True)
data.generate_dataset(HDATA / "val", 100, 2_000_000, cfg_h, with_depth=True)

cfg_train = lt.TrainConfig(
    train_dirs=[str(HDATA / "train")], val_dir=str(HDATA / "val"), out_dir="/content/runs/hybrid",
    targets=["hybrid", "layer"], margin=MARGIN, steps=1500, batch_size=8, warmup=100,
    eval_every=500, save_every=500, num_workers=min(4, os.cpu_count()),
)
history = lt.train(cfg_train)
single = lm.load_model("/content/runs/hybrid/best")

In [ ]:
for i, (rgb, lab, dep) in enumerate(BENCH):
    T = fusion.hybrid_target(dep, lab, MARGIN)
    out = lm.predict(single, rgb, size=INFER_SIZE, flip_tta=False)
    bench_rows[i]["single model (hybrid channel)"] = hybrid_metrics(out[0], T, dep, lab)
    bench_rows[i]["single model (layer channel)"] = hybrid_metrics(out[1], T, dep, lab)
table = summarize(bench_rows)
keep = [r for r in table.index if "|" not in r or r.split(" | ")[0] in ("multiply", "stacked", "ordered")]
table.loc[keep]

In [ ]:
for p, r in part1.items():
    out = lm.predict(single, r["rgb"], size=INFER_SIZE)
    r["single"], r["single_layers"] = out[0], layers.depth_to_layers(out[1], N_ID_LAYERS, rgb=r["rgb"])
    show_row([r["rgb"], layers.colorize_depth(r["ordered"]), layers.colorize_depth(out[0]),
              layers.colorize(r["single_layers"])],
             [p.name, "fusion: ordered (2 models)", "single model: hybrid", "single model: layers"], size=3.5)
    plot_profiles({"Depth Anything": r["depth"], "fusion: ordered": r["ordered"], "single model": out[0]},
                  r["labels"], p.name)

## 5. Parallax check (the eye test)

In [ ]:
PICK = paintings[0]  # change to any painting
r = part1[PICK]
gif_dir = Path("/content/parallax"); gif_dir.mkdir(exist_ok=True)
for name in ("depth", "stacked", "ordered", "single"):
    path = viz.save_gif(viz.parallax_frames(r["rgb"], r[name]), gif_dir / f"{PICK.stem}_{name}.gif")
    print(name); display(IPImage(filename=str(path)))

## 6. What to look for and what to do next

* **Part 1 vs Part 2 on the benchmark:** with *oracle* layers the fusions bound what a perfect layer predictor could
  give; with *noisy* / *ID* layers they show what you actually get. The single model has no separate layer input, so
  compare it against the noisy / ID rows.
* **`rho_within`** is the road/plain criterion. Fusions inherit Depth Anything's inside-layer depth, so they cannot
  beat it; only a model trained on painterly data can.
* **Synthetic numbers flatter the single model** because it is trained on the same generator. Real paintings
  (sections 2 and 5) are the check that matters. Label a few with true layers *and* draw the expected depth direction
  before trusting either approach.
* If the single model wins on paintings, scale it up with `02_train_full.ipynb` using
  `targets=["hybrid", "layer"]` and data generated with `with_depth=True`.